## Metric & differential forms setup

In [ ]:
from sympy import symbols, Matrix, sin, simplify
import numpy as np
from riemannian import (
    Metric, hodge_star, hodge_decomposition,
    de_rham_laplacian, verify_gauss_bonnet,
    RiemannianGrid, analyze_hodge_decomposition
)

# ── coordinates ──────────────────────────────────────────
x, y         = symbols('x y',         real=True)
theta, phi   = symbols('theta phi',   real=True)

# ── flat torus  ds² = dx² + dy² ──────────────────────────
m_flat  = Metric(Matrix([[1, 0], [0, 1]]), (x, y))

# ── Poincaré half-plane  ds² = (dx²+dy²)/y² ─────────────
m_hyp   = Metric(Matrix([[1/y**2, 0], [0, 1/y**2]]), (x, y))

# ── unit sphere  ds² = dθ² + sin²θ dφ² ──────────────────
m_sph   = Metric(Matrix([[1, 0], [0, sin(theta)**2]]),
                  (theta, phi))

## The Hodge star operator ⋆

In [ ]:
# ── Hodge star on the sphere ──────────────────────────────
star0 = hodge_star(m_sph, form_degree=0)
star1 = hodge_star(m_sph, form_degree=1)
star2 = hodge_star(m_sph, form_degree=2)

# ⋆1 = dV = sin θ  (volume form on S²)
print(simplify(star0(1)))

# ⋆(dθ) = sin θ dφ  — rotates the orthonormal frame by 90°
beta_x, beta_y = star1(1, 0)
print(simplify(beta_x), simplify(beta_y))

# ⋆⋆α = (−1)^{k(n−k)} α  →  ⋆⋆ = id on 1-forms in 2D
alpha_x, alpha_y = symbols('a b')
round_trip = star1(*star1(alpha_x, alpha_y))
print([simplify(c) for c in round_trip])   # → [a, b]

## de Rham Laplacian & the Weitzenböck identity

In [ ]:
# ── 0-form Laplacian (= Laplace–Beltrami) ────────────────
op0 = de_rham_laplacian(m_flat, form_degree=0)
from sympy import sin as Sin, cos
f = Sin(x) * cos(y)
print(simplify(op0['action'](f)))   # −2 sin(x)cos(y)

# ── 1-form Laplacian on the sphere (K = 1) ───────────────
op1_sph = de_rham_laplacian(m_sph, form_degree=1)
print(simplify(op1_sph['weitzenbock']))  # K = 1

# Apply Δ to the 1-form α = sin θ dθ  (i.e. α = (sin θ, 0))
result = op1_sph['action']((sin(theta), 0))
print([simplify(c) for c in result])

# ── Principal symbol  σ₂(Δ)(x,ξ) = gⁱʲ ξᵢ ξⱼ ──────────
op1_hyp = de_rham_laplacian(m_hyp, form_degree=1)
print(op1_hyp['principal'])   # y²(ξ² + η²)
print(simplify(op1_hyp['weitzenbock']))  # K = −1

## Hodge decomposition  α = dφ + ⋆dψ + h

In [ ]:
# ── Decompose the rotation form α = −y dx + x dy ─────────
# On the flat torus this form is harmonic: it generates H¹_dR.
domain = ((0, 2*np.pi), (0, 2*np.pi))

dec = hodge_decomposition(
    m_flat,
    omega_components=(-y, x),       # (αₓ, αᵧ) as SymPy expressions
    domain=domain,
    resolution=60,
)

# After the decomposition call
grid = dec['grid']

# Reconstruct original from decomposition (optional)
alpha_x = dec['alpha_exact'][0] + dec['alpha_coexact'][0] + dec['alpha_harmonic'][0]
alpha_y = dec['alpha_exact'][1] + dec['alpha_coexact'][1] + dec['alpha_harmonic'][1]

# L² norm (Euclidean sum)
norm2 = lambda u, v: np.sum(u**2 + v**2)
E_tot = norm2(alpha_x, alpha_y)

frac = lambda u, v: 100 * norm2(u, v) / (E_tot + 1e-30)
print(f"exact   {frac(dec['alpha_exact'][0], dec['alpha_exact'][1]):.1f}%")
print(f"co-exact{frac(dec['alpha_coexact'][0], dec['alpha_coexact'][1]):.1f}%")
print(f"harmonic{frac(dec['alpha_harmonic'][0], dec['alpha_harmonic'][1]):.1f}%")
# ----------------------------------------------------------------------
# 4. Analyze and visualise
# ----------------------------------------------------------------------
analyze_hodge_decomposition(dec, original=(-y, x), show_plot=True)

## RiemannianGrid — FEM matrices for both form degrees

In [ ]:
# ── Inspect matrices returned by the decomposition ───────
print(grid.A_scalar.shape)   # (3600, 3600) for resolution=60
print(grid.A_1form.shape)    # (7200, 7200)

# ── Build a grid directly and solve Δ₁ α = f ─────────────
grid_hyp = RiemannianGrid(m_hyp, ((0.1, 2), (0.5, 2.5)), resolution=60)

# Right-hand side: f = (sin(πx/2), 0) as numpy arrays
rhs = np.stack([
    np.sin(np.pi * grid_hyp.X / 2),
    np.zeros_like(grid_hyp.X),
])
sol = grid_hyp.solve_poisson_neumann(rhs)   # shape (2, 40, 40)
print(sol.shape)             # (2, 40, 40)

# ── Verify harmonic part satisfies Δ₁ h ≈ 0 ─────────────
h_vec = np.stack([dec['alpha_harmonic'][0].ravel(), dec['alpha_harmonic'][1].ravel()])  # from §4 decomposition
residual = grid.A_1form.dot(h_vec.ravel())
print(f"‖Δ₁h‖∞ = {np.abs(residual).max():.2e}")  # ≈ 0  (harmonic by construction)

## Gauss–Bonnet verification

In [ ]:
# ── Unit sphere: χ = 2, so ∫K dA = 4π ───────────────────
gb_sph = verify_gauss_bonnet(
    m_sph,
    domain=((0.05, np.pi-0.05), (0, 2*np.pi)),   # avoid poles
)
print(f"∫K dA = {gb_sph['integral']:.6f}")       # ≈ 4π ≈ 12.566
print(f"error  = {gb_sph['relative_error']:.2e}")

# ── Poincaré half-plane: K = −1 everywhere ───────────────
gb_hyp = verify_gauss_bonnet(
    m_hyp,
    domain=((-1, 1), (0.5, 2.5)),
)
print(f"∫K dA = {gb_hyp['integral']:.6f}")       # negative (K = −1)

# ── Flat metric: K = 0 everywhere ────────────────────────
gb_flat = verify_gauss_bonnet(m_flat, domain=((0,1),(0,1)))
print(f"∫K dA = {gb_flat['integral']:.2e}")          # ≈ 0

## 2-form decomposition — Helmholtz on a curved surface

In [ ]:
# ── Vorticity field ω = sin(πx)sin(πy) dx∧dy ─────────────
# This is a localised vortex patch on the flat plane.
from sympy import sin, pi

domain = ((0, 1), (0, 1))
omega_coeff = sin(pi * x) * sin(pi * y)   # f(x,y)

dec2 = hodge_decomposition(
    m_flat, omega_coeff, domain,
    resolution=60, form_degree=2,
)

w_ex  = dec2['omega_exact']     # irrotational part
w_ha  = dec2['omega_harmonic']  # global circulation (b₂=0 → ≈ 0)

# ── Energy fractions ─────────────────────────────────────
grid  = dec2['grid']
e     = lambda a: (a**2).sum()
E     = e(w_ex) + e(w_ha) + 1e-30
print(f"irrotational  {100*e(w_ex)/E:.1f}%")
print(f"harmonic      {100*e(w_ha)/E:.1f}%")  # ≈ 0 (contractible domain)

# ── Same form on the Poincaré half-plane ─────────────────
# Curvature K = −1 warps the decomposition: the metric weight
# √|g| = 1/y² concentrates energy near y = 0.
dec2_hyp = hodge_decomposition(
    m_hyp, 1, ((-1, 1), (0.5, 2.5)),
    resolution=50, form_degree=2,
)
print("half-plane ω_harmonic max:",
      np.abs(dec2_hyp['omega_harmonic']).max())

analyze_hodge_decomposition(dec2, original=omega_coeff, show_plot=True)

## Detecting topology — mixed form on the flat torus

In [ ]:
from sympy import sin, cos

# ── Build α = exact + co-exact + c·dx ────────────────────
# Exact part:    d(sin x sin y) = cos(x)sin(y) dx + sin(x)cos(y) dy
ex_x_sym  =  cos(x) * sin(y)
ex_y_sym  =  sin(x) * cos(y)

# Co-exact part: ⋆d(cos x cos y) on flat metric
#   d(cos x cos y) = −sin(x)cos(y) dx − cos(x)sin(y) dy
#   ⋆(a dx + b dy) = −b dx + a dy
co_x_sym  =  cos(x) * sin(y)     # = −(−cos x sin y)
co_y_sym  = -sin(x) * cos(y)

# Harmonic generator of H¹_dR(T²): the constant 1-form dx (i.e. (1, 0))
c         = 3.0                       # amplitude of harmonic generator

alpha_x   = ex_x_sym + co_x_sym + c  # total αₓ
alpha_y   = ex_y_sym + co_y_sym      # total αᵧ

dec_mix = hodge_decomposition(
    m_flat, (alpha_x, alpha_y),
    domain=((0, 2*np.pi), (0, 2*np.pi)), resolution=60,
)

ha_x, ha_y = dec_mix['alpha_harmonic']

# ── The harmonic part should recover c·dx = (3, 0) ───────
# In the interior, ha_x ≈ 3 and ha_y ≈ 0 everywhere.
sl = np.s_[5:-5, 5:-5]   # interior, away from Dirichlet boundary
print(f"ha_x interior mean: {ha_x[sl].mean():.3f}")  # ≈ 3.0
print(f"ha_y interior mean: {ha_y[sl].mean():.3f}")  # ≈ 0.0

# ── Vary c and watch the harmonic amplitude track it ─────
for c_val in [0.5, 1.0, 2.0, 4.0]:
    d = hodge_decomposition(
        m_flat,
        (ex_x_sym + co_x_sym + c_val, ex_y_sym + co_y_sym),
        ((0, 2*np.pi), (0, 2*np.pi)), resolution=40,
    )
    h = d['alpha_harmonic'][0][sl].mean()
    print(f"  c={c_val:.1f}  →  ha_x ≈ {h:.2f}")
    analyze_hodge_decomposition(d, original=(ex_x_sym + co_x_sym + c_val, ex_y_sym + co_y_sym), show_plot=True)

## Bochner's theorem — harmonic 1-forms vanish on S²

In [ ]:
# ── Symbolic confirmation: K = 1 > 0 on S² ───────────────
op_sph = de_rham_laplacian(m_sph, form_degree=1)
print("Weitzenböck K =", simplify(op_sph['weitzenbock']))   # 1

# ── Attempt to decompose a non-trivial 1-form on S² ──────
# α = sin θ dθ  looks like a "rotation generator" but is harmonic.
# On S², Bochner forces the harmonic part to be zero.
dec_sph = hodge_decomposition(
    m_sph,
    (sin(theta), 0),
    domain=((0.1, np.pi-0.1), (0, 2*np.pi)),
    resolution=50,
)
ha_x, ha_y = dec_sph['alpha_harmonic']
norm_ha = np.sqrt((ha_x**2 + ha_y**2).sum())
norm_alpha = np.sqrt((dec_sph['alpha_exact'][0]**2
              + dec_sph['alpha_coexact'][0]**2).sum() + 1e-30)
print(f"‖harmonic‖ / ‖α‖  = {norm_ha/norm_alpha:.4f}")   # ≈ 0

# ── Contrast with the flat torus (K = 0, b₁ = 2) ────────
dec_tor = hodge_decomposition(
    m_flat, (1, 0),   # constant 1-form dx — purely harmonic on T²
    domain=((0, 2*np.pi), (0, 2*np.pi)), resolution=50,
)
ha_tor = dec_tor['alpha_harmonic']
norm_tor = np.sqrt((ha_tor[0]**2 + ha_tor[1]**2).sum())
norm_in  = np.sqrt((1.0)**2 * dec_tor['grid'].N**2)
print(f"‖harmonic‖ / ‖α‖  = {norm_tor/norm_in:.4f}")   # ≈ 1 (entirely harmonic)
analyze_hodge_decomposition(dec_tor, original=(1, 0), show_plot=True)

# ── Verify Δ₁(sin θ dθ) = 0 symbolically ────────────────
result = op_sph['action']((sin(theta), 0))
print("Δ₁(sin θ dθ) =", [simplify(c) for c in result])   # [0, 0]

## Poincaré duality — ⋆ exchanges harmonic generators

In [ ]:
# ── Symbolic: ⋆(dx) = dy on the flat torus ───────────────
star1_flat = hodge_star(m_flat, form_degree=1)
bx, by     = star1_flat(1, 0)     # ⋆(1·dx + 0·dy)
print(simplify(bx), simplify(by))  # 0, 1  → ⋆dx = dy  ✓

cx, cy     = star1_flat(0, 1)     # ⋆(0·dx + 1·dy)
print(simplify(cx), simplify(cy)) # -1, 0 → ⋆dy = −dx ✓

# ── Numerical: decompose dx, extract harmonic, apply ⋆ ───
domain_torus = ((0, 2*np.pi), (0, 2*np.pi))

# Generator [dx]: harmonic part of the constant 1-form (1, 0)
dec_dx  = hodge_decomposition(m_flat, (1, 0), domain_torus, resolution=50)
h_dx_x, h_dx_y = dec_dx['alpha_harmonic']

# Apply ⋆ numerically to the extracted harmonic (h_dx_x, h_dx_y)
grid    = dec_dx['grid']
g00     = grid.g_inv00;  g11 = grid.g_inv11;  g01 = grid.g_inv01
sd      = grid.sqrt_det
# ⋆(a dx + b dy) on flat metric: (−b, a)·√g  with √g=1
star_hx = (g00 * h_dx_y - g01 * h_dx_x) * sd
star_hy = (g11 * h_dx_x - g01 * h_dx_y) * sd

# Decompose the result: its harmonic part should be ≈ [dy] = (0, 1)
dec_star = hodge_decomposition(m_flat, (lambda x,y: star_hx.mean(),
                                           lambda x,y: star_hy.mean()),
                               domain_torus, resolution=50)
sl = np.s_[5:-5, 5:-5]
h2_x, h2_y = dec_star['alpha_harmonic']
print(f"⋆[dx] harmonic: ({h2_x[sl].mean():.3f}, {h2_y[sl].mean():.3f})")  # ≈ (0, 1) = [dy]

## Heat flow on forms — a dynamical proof of Hodge

In [ ]:
from scipy.sparse.linalg import expm_multiply

# ── Build A_1form on the flat torus ───────────────────────
# grid.A_1form = Δ₁ (negative semi‑definite on the orthogonal complement of harmonic forms)
domain_torus = ((0, 2*np.pi), (0, 2*np.pi))
grid = RiemannianGrid(m_flat, domain_torus, resolution=40)
L = grid.A_1form   # Δ₁

# ── Initial condition: mixed form α₀ = (sin x, cos y) ────
# This contains no harmonic part on a contractible domain, so it decays to zero.
N = grid.N
X, Y = grid.X, grid.Y
a0_x = np.sin(X).ravel()
a0_y = np.cos(Y).ravel()
u0 = np.concatenate([a0_x, a0_y])

# ── Evolve: α(t) = exp(t Δ₁) α₀ ─────────────────────────
print("Decay of non‑harmonic part:")
for t in [0.0, 0.05, 0.2, 1.0, 5.0, 20.0]:
    ut = expm_multiply(t * L, u0)   # note: t*L, not -t*L
    norm = np.linalg.norm(ut)
    print(f"t={t:5.2f}  ‖α(t)‖ = {norm:.4f}")

# ── Now with a harmonic initial condition: (1, 0) ────────
# Harmonic forms are in the kernel of Δ₁, so they stay unchanged.
u_harm = np.concatenate([
    np.full(N**2, 1.0),   # αₓ = 1 everywhere
    np.zeros(N**2),       # αᵧ = 0
])
print("\nPreservation of harmonic part:")
for t in [0, 1, 10, 100]:
    ut = expm_multiply(t * L, u_harm)
    # Since ut is still harmonic, L·ut should be zero (up to numerical error)
    residual_norm = np.linalg.norm(L.dot(ut), ord=np.inf)
    print(f"t={t:4d}  ‖Δ₁α(t)‖∞ = {residual_norm:.2e}  (should be ≈ 0)")

## Magnetic monopoles — topology obstructs a vector potential

In [ ]:
# ── The "monopole" 1-form on the punctured plane ─────────
# α = (−y dx + x dy) / (x²+y²)  is the angle form dθ.
# It is closed (dα = 0) but NOT exact: ∮ α = 2π.
# The Hodge decomposition on a domain with boundary uses:
#   φ with Dirichlet BC (φ=0 on boundary)
#   ψ with Neumann BC (∂ₙψ=0 on boundary)
# This splitting leads to a harmonic part h that is not the unique
# cohomology representative but rather a different harmonic form
# satisfying those boundary conditions. For the angle form,
# the decomposition yields h ≈ 2 dθ, and the co‑exact part ⋆dψ ≈ −dθ.
# Consequently, the line integral of the harmonic part around the
# inner circle is 4π instead of 2π. The winding number (topological
# charge) of the original form is still 2π, and can be obtained
# directly from α.

# Work in an annular region to avoid the singularity at origin.
r_min, r_max = 0.3, 2.0
domain_ann   = ((-r_max, r_max), (-r_max, r_max))

# Angle form: αₓ = −y/(x²+y²),  αᵧ = x/(x²+y²)
def angle_x(x, y): return -y / (x**2 + y**2 + 1e-8)
def angle_y(x, y): return  x / (x**2 + y**2 + 1e-8)

# Perform the Hodge decomposition (using Dirichlet φ, Neumann ψ)
dec_mono = hodge_decomposition(
    m_flat, (angle_x, angle_y), domain_ann, resolution=60,
)
ha_x, ha_y = dec_mono['alpha_harmonic']
ex_x, ex_y = dec_mono['alpha_exact']
co_x, co_y = dec_mono['alpha_coexact']

analyze_hodge_decomposition(dec_mono, original=(angle_x, angle_y), show_plot=True)
# Energy distribution — the harmonic part carries most of the energy,
# but note that the exact part is zero because δα=0 and φ=0 on boundary.
e    = lambda u, v: (u**2 + v**2).sum()
E    = e(ha_x, ha_y) + e(ex_x, ex_y) + e(co_x, co_y) + 1e-30
print(f"harmonic part energy: {100*e(ha_x, ha_y)/E:.1f}%")
print(f"exact part energy:    {100*e(ex_x, ex_y)/E:.1f}%")
print(f"co‑exact part energy: {100*e(co_x, co_y)/E:.1f}%")
# Output example: harmonic ≈ 81.6%, exact ≈ 0.0%, co‑exact ≈ 18.4%

# ── Compute the winding number from the original α ─────────
# This gives the true topological charge, independent of the decomposition.
grid   = dec_mono['grid']
t_circ = np.linspace(0, 2*np.pi, 500)
xc, yc = np.cos(t_circ), np.sin(t_circ)    # unit circle
# Interpolate original α onto the circle
from scipy.interpolate import RegularGridInterpolator
xs = np.linspace(domain_ann[0][0], domain_ann[0][1], grid.N)
ys = np.linspace(domain_ann[1][0], domain_ann[1][1], grid.N)
itp_x = RegularGridInterpolator((xs, ys), angle_x(grid.X, grid.Y))
itp_y = RegularGridInterpolator((xs, ys), angle_y(grid.X, grid.Y))
pts   = np.column_stack([xc, yc])
dx_dt  = -np.sin(t_circ);  dy_dt = np.cos(t_circ)
integrand = itp_x(pts) * dx_dt + itp_y(pts) * dy_dt
winding   = np.trapezoid(integrand, t_circ)
print(f"∮ original α = {winding:.4f}")   # ≈ 2π ≈ 6.2832 — topological charge = 1

# ── (Optional) Compute circulation of the harmonic part ───
# Because of the boundary conditions, this gives 4π, not 2π.
itp_ha_x = RegularGridInterpolator((xs, ys), ha_x)
itp_ha_y = RegularGridInterpolator((xs, ys), ha_y)
integrand_h = itp_ha_x(pts) * dx_dt + itp_ha_y(pts) * dy_dt
winding_h   = np.trapezoid(integrand_h, t_circ)
print(f"∮ harmonic part = {winding_h:.4f}   (double due to BCs)")   # ≈ 4π ≈ 12.5664

## Conformal invariance — ⋆ on 1-forms sees only angles

In [ ]:
# ── Numerical demonstration: conformal invariance of the harmonic part ─────
# In 2D, the harmonic part of a fixed 1‑form is unchanged when the metric is
# conformally rescaled (even though the exact and coexact parts adjust).
# We test this with three conformally equivalent metrics.

from sympy import symbols, Matrix, sin, exp
import numpy as np
from riemannian import Metric, hodge_decomposition

x, y = symbols('x y', real=True)

# Domain: a rectangle away from the zeros of sin(x) (to keep the metric positive)
domain = ((0.5, np.pi - 0.5), (0.5, np.pi - 0.5))

metrics = [
    ("flat",      Matrix([[1, 0], [0, 1]])),
    ("4·flat",    Matrix([[4, 0], [0, 4]])),
    ("e^{2sinx}", Matrix([[exp(2*sin(x)), 0], [0, exp(2*sin(x))]])),
]

harmonic_means = []

for name, g in metrics:
    m = Metric(g, (x, y))
    dec = hodge_decomposition(m, (1, 0), domain, resolution=60)
    # Harmonic part is (h_x, h_y); take the x‑component and average interior points
    hx = dec['alpha_harmonic'][0]
    avg = hx[5:-5, 5:-5].mean()   # avoid boundary artifacts
    harmonic_means.append(avg)
    print(f"{name:15s}   harmonic_x ≈ {avg:.5f}")
    analyze_hodge_decomposition(dec, original=(1, 0), show_plot=True)
    

print(f"\nMaximum variation: {max(harmonic_means) - min(harmonic_means):.5f}")
# Expected output: all three ≈ 1.00000, variation < 1e‑4

## Spectrum of the Laplacian — hearing the shape of a manifold

In [ ]:
from scipy.sparse.linalg import eigsh
from scipy.sparse import csr_matrix

# ── Flat square [0, π]² with Dirichlet BC ─────────────────
grid_sq = RiemannianGrid(m_flat, ((0, np.pi), (0, np.pi)), resolution=40)

# Extract interior nodes
src = np.arange(grid_sq.N2).reshape(grid_sq.N, grid_sq.N)
boundary = np.zeros_like(src, dtype=bool)
boundary[0, :] = boundary[-1, :] = boundary[:, 0] = boundary[:, -1] = True
interior = src[~boundary].flatten()

# A_scalar = −Δ; interior submatrix is symmetric and negative definite.
A_int = grid_sq.A_scalar[interior, :][:, interior].tocsc()

# Compute the smallest eigenvalues of A_int (they are the negatives of the
# true Dirichlet Laplacian eigenvalues). So we take -λ to get positive λ.
vals_neg, _ = eigsh(A_int, k=10, which='SM')
vals_sq = np.sort(-vals_neg)            # positive Dirichlet eigenvalues

# Exact eigenvalues on [0,π]²: λ_{m,n} = m² + n², m,n ≥ 1
exact = np.array(sorted(m**2 + n**2 for m in range(1, 6) for n in range(1, 6)))[:10]

print("Flat square — first 10 Dirichlet eigenvalues:")
print("Index   Numerical   Exact")
for i, (num, ex) in enumerate(zip(vals_sq[:10], exact[:10]), 1):
    print(f"{i:3d}    {num:8.3f}    {ex:5d}")

# ── Poincaré half‑plane ℍ²  K = −1 with Dirichlet BC ──────
grid_hyp = RiemannianGrid(m_hyp, ((-2, 2), (0.5, 3.0)), resolution=35)
src_hyp = np.arange(grid_hyp.N2).reshape(grid_hyp.N, grid_hyp.N)
boundary_hyp = np.zeros_like(src_hyp, dtype=bool)
boundary_hyp[0, :] = boundary_hyp[-1, :] = boundary_hyp[:, 0] = boundary_hyp[:, -1] = True
interior_hyp = src_hyp[~boundary_hyp].flatten()
A_int_hyp = grid_hyp.A_scalar[interior_hyp, :][:, interior_hyp].tocsc()

# Compute smallest eigenvalues of the negative Laplacian, then flip sign
vals_neg_hyp, _ = eigsh(A_int_hyp, k=8, which='SM')
vals_hyp = np.sort(-vals_neg_hyp)       # positive Dirichlet eigenvalues

print("\nPoincaré half-plane — first 6 eigenvalues (Dirichlet):")
print("  numerical:", np.round(vals_hyp[:6], 3))
print(f"  λ_min = {vals_hyp[0]:.4f}  (theory: λ ≥ 1/4 = {0.25:.4f})")

# ── Weyl's law: N(λ) ≈ Area(M)·λ / (4π) ─────────────────
area_sq = np.pi**2
lam = 10.0
N_count = (vals_sq <= lam).sum()
N_weyl = area_sq * lam / (4 * np.pi)
print(f"\nWeyl count N(10): numerical={N_count},  Weyl≈{N_weyl:.1f}")